# EXP-007 — synthetic physics-prior credibility feasibility

This is a thin, restartable controller for the locked EXP-007 decision gate. It trains a small causal data-only backbone, cross-fits label-safe credibility evidence, freezes and calibrates the credibility estimator on training/validation trajectories, and evaluates the untouched controlled test trajectories once.

## 1. Mount Drive and verify the exact pushed source

Upload the newly prepared `Upload` folder to `MyDrive`, open this notebook from that folder, select a **T4 GPU**, and run all cells. The commit is read from `expected_commit.txt`; no SHA editing is required.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, shutil, subprocess, sys
from pathlib import Path

UPLOAD = Path("/content/drive/MyDrive/Upload")
FEATURE_PATH = UPLOAD / "feature_cache" / "controlled_synthetic_features.csv"
COMMIT_FILE = UPLOAD / "expected_commit.txt"
REPOSITORY_URL = "https://github.com/mahdeehassansami/Thesis-Physics-Informend-Neural-Network.git"
if not FEATURE_PATH.is_file():
    raise FileNotFoundError(f"Missing EXP-007 feature cache: {FEATURE_PATH}")
if not COMMIT_FILE.is_file():
    raise FileNotFoundError(f"Missing automatic commit pin: {COMMIT_FILE}")
EXPECTED_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip().lower()
if len(EXPECTED_COMMIT) != 40 or any(ch not in "0123456789abcdef" for ch in EXPECTED_COMMIT):
    raise ValueError("expected_commit.txt must contain one full lowercase Git SHA.")

CLONE = Path("/content/thesis_work_exp007")
if CLONE.exists():
    shutil.rmtree(CLONE)
subprocess.run(["git", "clone", "--quiet", REPOSITORY_URL, str(CLONE)], check=True)
subprocess.run(["git", "checkout", "--quiet", EXPECTED_COMMIT], cwd=CLONE, check=True)
actual = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CLONE, check=True, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=CLONE, check=True, capture_output=True, text=True).stdout.strip()
if actual != EXPECTED_COMMIT or dirty:
    raise RuntimeError(f"Unreproducible checkout: expected={EXPECTED_COMMIT}, actual={actual}, dirty={bool(dirty)}")
os.chdir(CLONE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CLONE / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(CLONE / "src"))
import thesis_work
print("Committed source:", actual)
print("thesis_work package:", Path(thesis_work.__file__).resolve())

## 2. Verify T4/CUDA, configuration, data identity, and recovery state

The cell stops before training if the assigned GPU, Git commit, controlled-cache fingerprint, immutable split, or recovery identity is wrong.

In [ ]:
subprocess.run(["nvidia-smi"], check=True)

import torch
from thesis_work.exp7_credibility import load_exp7_config, validate_exp7_runtime

CONFIG_PATH = CLONE / "configs" / "experiment.yaml"
config = load_exp7_config(CONFIG_PATH)
config["repository"]["expected_commit"] = EXPECTED_COMMIT
environment, git, qualification = validate_exp7_runtime(config, CLONE, FEATURE_PATH)
WORK_DIR = Path(config["runtime"]["train_work_directory"])
DRIVE_OUTPUT = Path(config["runtime"]["drive_output_directory"])
if DRIVE_OUTPUT.exists() and any(DRIVE_OUTPUT.iterdir()) and not (DRIVE_OUTPUT / "run_state.json").exists():
    raise FileExistsError("EXP-007 Drive output is nonempty without a compatible run_state.json.")
if not WORK_DIR.exists() and (DRIVE_OUTPUT / "run_state.json").exists():
    shutil.copytree(DRIVE_OUTPUT, WORK_DIR)
print(json.dumps({"experiment": config["experiment"], "environment": environment, "git": git, "data": qualification, "work": str(WORK_DIR), "recovery": str(DRIVE_OUTPUT)}, indent=2))

## 3. Run the five-seed cross-fitted feasibility experiment

Training occurs under `/content` for speed. Completed folds and seeds are synchronized to Drive. Test labels do not fit normalization, templates, the backbone, the degradation proxy, calibration, threshold, or scalar comparator.

In [ ]:
from thesis_work.exp7_credibility import run_exp7_experiment

gate = run_exp7_experiment(
    config=config,
    project_root=CLONE,
    feature_path=FEATURE_PATH,
    output_root=WORK_DIR,
    recovery_root=DRIVE_OUTPUT,
)
print(json.dumps(gate, indent=2))

## 4. Inspect the predeclared decision gate

Proceed only if held-out trajectory-candidate AUROC is at least 0.80, its trajectory/seed bootstrap 95% lower bound is above 0.50, and credibility has not collapsed. A failed gate is a scientific result and must not be repaired by test-driven tuning.

In [ ]:
import pandas as pd

display(pd.read_csv(WORK_DIR / "credibility_metrics.csv"))
display(pd.read_csv(WORK_DIR / "model_comparison.csv"))
display(pd.read_csv(WORK_DIR / "physics_regret.csv").groupby(["method", "validity_label"], as_index=False)["physics_regret"].mean())
print((WORK_DIR / "summary.md").read_text(encoding="utf-8"))

## 5. Finalize the full Drive artifacts and lightweight Codex bundle

Save this notebook before running the cell. The complete output retains checkpoints; the ZIP excludes checkpoint/model binary files.

In [ ]:
from thesis_work.exp7_credibility import finalize_exp7_artifacts

notebook_source = UPLOAD / "train_models_colab.ipynb"
if notebook_source.is_file():
    shutil.copy2(notebook_source, WORK_DIR / "executed_notebook.ipynb")
bundle = finalize_exp7_artifacts(WORK_DIR)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
shutil.copytree(WORK_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
manifest = json.loads((DRIVE_OUTPUT / "run_manifest.json").read_text(encoding="utf-8"))
decision = json.loads((DRIVE_OUTPUT / "gate_decision.json").read_text(encoding="utf-8"))
print(json.dumps({"status": manifest["status"], "gate_decision": decision["decision"], "full_output": str(DRIVE_OUTPUT), "bundle": str(DRIVE_OUTPUT / bundle.name), "next_step": "Download experiment_outputs_exp007 and place it in thesis-work/results/incoming for Codex validation."}, indent=2))